In [ ]:
import pandas as pd
from scipy import stats
import numpy as np

# ========== Data Preprocessing ==========
df = pd.read_excel(
    r"D:\...\Combined_NegEvtRatios_Joined_20251119.xlsx",
    engine='openpyxl'
)

# Filter valid data with non-null Ratio values
df_valid = df.dropna(subset=['Ratio']).copy()
print(f"Total valid rows (non-null Ratio): {len(df_valid)}")

# ========== Group Definitions ==========
# 1. SIDS / Non-SIDS groups
df_valid['SIDS_Label'] = df_valid['SIDS'].astype(int).map({1: 'SIDS', 0: 'Non-SIDS'})
sids_data = df_valid[df_valid['SIDS_Label'] == 'SIDS']['Ratio']
non_sids_data = df_valid[df_valid['SIDS_Label'] == 'Non-SIDS']['Ratio']
print(f"\nSIDS group size: {len(sids_data)}")
print(f"Non-SIDS group size: {len(non_sids_data)}")

# 2. LDC / DC / Non-LDC-DC groups
df_valid['LDC_Label'] = df_valid['LDC'].astype(int).map({1: 'LDC', 2: 'DC', 0: 'Non-LDC-DC'})
ldc_data = df_valid[df_valid['LDC_Label'] == 'LDC']['Ratio']
dc_data = df_valid[df_valid['LDC_Label'] == 'DC']['Ratio']
non_ldc_dc_data = df_valid[df_valid['LDC_Label'] == 'Non-LDC-DC']['Ratio']
print(f"\nLDC group size: {len(ldc_data)}")
print(f"DC group size: {len(dc_data)}")
print(f"Non-LDC-DC group size: {len(non_ldc_dc_data)}")

# 3. NDGAIN Top 25% / Bottom 75% groups
df_ndgain_valid = df_valid.dropna(subset=['NDGAINScore']).copy()
NDGAINScore_75p = np.percentile(df_ndgain_valid['NDGAINScore'], 75)
df_ndgain_valid['NDGAIN_Label'] = np.where(
    df_ndgain_valid['NDGAINScore'] >= NDGAINScore_75p,
    'NDGAIN Top 25%',
    'NDGAIN Bottom 75%'
)
ndgain_top_data = df_ndgain_valid[df_ndgain_valid['NDGAIN_Label'] == 'NDGAIN Top 25%']['Ratio']
ndgain_bottom_data = df_ndgain_valid[df_ndgain_valid['NDGAIN_Label'] == 'NDGAIN Bottom 75%']['Ratio']
print(f"\nNDGAIN Top 25% group size: {len(ndgain_top_data)}")
print(f"NDGAIN Bottom 75% group size: {len(ndgain_bottom_data)}")

# ========== Statistical Test Function ==========
def group_test(group1_data, group2_data, group1_name, group2_name):
    # Normality test (K-S test)
    _, p_norm1 = stats.kstest(group1_data, 'norm')
    _, p_norm2 = stats.kstest(group2_data, 'norm')
    norm1 = p_norm1 > 0.05
    norm2 = p_norm2 > 0.05
    
    # Homogeneity of variance test
    _, p_levene = stats.levene(group1_data, group2_data)
    var_equal = p_levene > 0.05
    
    # Significance test selection
    if norm1 and norm2 and var_equal:
        _, p_diff = stats.ttest_ind(group1_data, group2_data, equal_var=True)
        test_method = "Independent t-test"
    else:
        _, p_diff = stats.mannwhitneyu(group1_data, group2_data)
        test_method = "Mann-Whitney U test"
    
    return (p_norm1, norm1, p_norm2, norm2, p_levene, var_equal, p_diff, test_method, group1_name, group2_name)

# ========== Run Statistical Tests ==========
# 1. SIDS vs Non-SIDS
sids_results = group_test(sids_data, non_sids_data, 'SIDS', 'Non-SIDS')

# 2. Pairwise comparisons for LDC groups
ldc_pairs = [
    (ldc_data, dc_data, 'LDC', 'DC'),
    (ldc_data, non_ldc_dc_data, 'LDC', 'Non-LDC-DC'),
    (dc_data, non_ldc_dc_data, 'DC', 'Non-LDC-DC')
]
ldc_results = [group_test(pair[0], pair[1], pair[2], pair[3]) for pair in ldc_pairs]

# 3. NDGAIN Top 25% vs Bottom 75%
ndgain_results = group_test(ndgain_top_data, ndgain_bottom_data, 'NDGAIN Top 25%', 'NDGAIN Bottom 75%')

# ========== Print Results ==========
def print_test_result(result):
    p_norm1, norm1, p_norm2, norm2, p_levene, var_equal, p_diff, method, g1, g2 = result
    print(f"【{g1} vs {g2}】")
    print(f"  {g1} normality test (K-S) p-value: {p_norm1:.3f}, normal distribution: {norm1}")
    print(f"  {g2} normality test (K-S) p-value: {p_norm2:.3f}, normal distribution: {norm2}")
    print(f"  Levene's test p-value: {p_levene:.3f}, equal variance: {var_equal}")
    print(f"  Test method: {method}, p-value: {p_diff:.3f}")
    
    if p_diff < 0.05:
        print(f"  Conclusion: Significant difference between groups (p<0.05)")
    else:
        print(f"  Conclusion: No significant difference between groups (p≥0.05)")

# 1. SIDS vs Non-SIDS
print("\n" + "="*60)
print("1. SIDS Group Comparison")
print("="*60)
print_test_result(sids_results)

# 2. LDC pairwise comparisons
print("\n" + "="*60)
print("2. LDC Group Pairwise Comparisons")
print("="*60)
for res in ldc_results:
    print_test_result(res)
    print("-"*40)

# 3. NDGAIN comparison
print("\n" + "="*60)
print("3. NDGAIN Group Comparison")
print("="*60)
print_test_result(ndgain_results)
print("="*60)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ========== 1. Load Data & Preprocessing ==========
df = pd.read_excel(
    r"E:\...\Combined_NegEvtRatios_Joined_20251119.xlsx",
    engine='openpyxl'
)

# Filter valid data with non-null Ratio
df_valid = df.dropna(subset=['Ratio']).copy()
print(f"Valid rows (non-null Ratio): {len(df_valid)}")

# 1. SIDS / Non-SIDS grouping
df_valid['SIDS_Label'] = df_valid['SIDS'].astype(int).map({1: 'SIDS', 0: 'Non-SIDS'})
sids_data = df_valid[df_valid['SIDS_Label'] == 'SIDS']['Ratio']
non_sids_data = df_valid[df_valid['SIDS_Label'] == 'Non-SIDS']['Ratio']
print(f"\nSIDS group size: {len(sids_data)}")
print(f"Non-SIDS group size: {len(non_sids_data)}")

# 2. LDC / DC / Non-LDC-DC grouping
df_valid['LDC_Label'] = df_valid['LDC'].astype(int).map({
    1: 'LDC',
    2: 'DC',
    0: 'Non-LDC-DC'
})
ldc_data = df_valid[df_valid['LDC_Label'] == 'LDC']['Ratio']
dc_data = df_valid[df_valid['LDC_Label'] == 'DC']['Ratio']
non_ldc_dc_data = df_valid[df_valid['LDC_Label'] == 'Non-LDC-DC']['Ratio']
print(f"\nLDC group size: {len(ldc_data)}")
print(f"DC group size: {len(dc_data)}")
print(f"Non-LDC-DC group size: {len(non_ldc_dc_data)}")

# 3. NDGAIN grouping
df_ndgain_valid = df_valid.dropna(subset=['NDGAINScore']).copy()
NDGAINScore_75p = np.percentile(df_ndgain_valid['NDGAINScore'], 75)
df_ndgain_valid['NDGAIN_Label'] = np.where(
    df_ndgain_valid['NDGAINScore'] >= NDGAINScore_75p,
    'NDGAIN Top 25%',
    'NDGAIN Bottom 75%'
)
ndgain_top_data = df_ndgain_valid[df_ndgain_valid['NDGAIN_Label'] == 'NDGAIN Top 25%']['Ratio']
ndgain_bottom_data = df_ndgain_valid[df_ndgain_valid['NDGAIN_Label'] == 'NDGAIN Bottom 75%']['Ratio']
print(f"\nNDGAIN Top 25% group size: {len(ndgain_top_data)}")
print(f"NDGAIN Bottom 75% group size: {len(ndgain_bottom_data)}")

# ========== 2. Global Plot Settings ==========
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(14, 4))
fig.subplots_adjust(
    left=0.07,
    bottom=0.15,
    right=0.96,
    top=0.85
)

# ========== 3. Define 7 Groups ==========
groups_info = [
    ('SIDS',            sids_data,          0,    '#0072BD'),
    ('Non-SIDS',        non_sids_data,      1,    '#4DBEEE'),
    ('LDC',             ldc_data,           2,    '#228B22'),
    ('DC',              dc_data,            3,    '#7CC767'),
    ('Non-LDC-DC',      non_ldc_dc_data,    4,    '#4CAF50'),
    ('NDGAIN Bottom 75%', ndgain_bottom_data, 5, '#FAA76C'),
    ('NDGAIN Top 25%',    ndgain_top_data,    6, '#D9531A'),
]

x_positions = [info[2] for info in groups_info]
group_names = [info[0] for info in groups_info]

# ========== 4. Draw Violin Plots ==========
for name, data, x_pos, color in groups_info:
    violin_plot = sns.violinplot(
        x=[x_pos]*len(data),
        y=data,
        hue=[x_pos]*len(data),
        ax=ax,
        inner=None,
        palette=[color],
        linewidth=0.5,
        linecolor='#827F82',
        width=0.6,
        cut=5,
        legend=False
    )

for patch in ax.patches:
    if 'Violin' in str(patch.__class__):
        patch.set_linestyle('--')

# ========== 5. Overlay Boxplots ==========
for name, data, x_pos, color in groups_info:
    sns.boxplot(
        x=[x_pos]*len(data),
        y=data,
        ax=ax,
        width=0.02,
        showfliers=False,
        boxprops={'color': '#827F82', 'zorder': 5},
        whiskerprops={'color': '#827F82', 'linestyle': '-'},
        capprops={'linewidth': 0},
        medianprops={'linewidth': 0},
        linecolor='#827F82'
    )

# ========== 6. Plot Median Circles ==========
medians = [data.median() for _, data, _, _ in groups_info]
ax.scatter(
    x=x_positions,
    y=medians,
    s=40,
    facecolors='white',
    edgecolors='#827F82',
    linewidths=1.5,
    zorder=10
)

# ========== 7. Axis Settings ==========
ax.set_ylim(-0.3, 1.3)
ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1])
ax.set_yticklabels([0, 0.2, 0.4, 0.6, 0.8, 1], fontsize=14)
ax.set_ylabel('Ratio', fontsize=14)

ax.set_xticks(x_positions)
ax.set_xticklabels(group_names, fontsize=14, rotation=0)
ax.set_xlim(-0.5, 6.5)

# Horizontal reference lines
for y_val in [0, 1]:
    ax.axhline(
        y=y_val,
        linestyle='--',
        linewidth=1.5,
        color=(165/255, 42/255, 42/255),
        zorder=3
    )

# Vertical dividers
ax.axvline(x=1.6, linestyle='--', color='#827F82', alpha=0.5, linewidth=1.5, zorder=2)
ax.axvline(x=4.6, linestyle='--', color='#827F82', alpha=0.5, linewidth=1.5, zorder=2)

# Frame style
for spine in ax.spines.values():
    spine.set_linewidth(1.2)
ax.tick_params(axis='both', width=1.2)

# ========== 9. Show & Save ==========
plt.show()

fig.savefig(
    r"C:\Users\Lenovo\Desktop\Violinplot.png",
    dpi=500,
    bbox_inches='tight'
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.transform import xy

# ==== 1. Font Settings ====
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 7

# ==== 2. Read .tif data ====
tif_path = r"E:\...\NeglectedEvents_251009.tif"

with rasterio.open(tif_path) as src:
    eta_data = src.read(1)
    transform = src.transform
    height, width = src.shape
    
    cols, rows = np.meshgrid(np.arange(width), np.arange(height))
    lon, lat = xy(transform, rows, cols)
    lon = np.array(lon)
    lat = np.array(lat)
    
    if src.nodata is not None:
        eta_data = np.where(eta_data == src.nodata, np.nan, eta_data)

lat_flat = lat.flatten()
eta_flat = eta_data.flatten()
valid_idx = ~np.isnan(eta_flat)
lat_flat = lat_flat[valid_idx]
eta_flat = eta_flat[valid_idx]

# ==== 3. Latitude interval statistics ====
n_intervals = 36
eta_lat = np.full(n_intervals, np.nan)
eta_std = np.full(n_intervals, np.nan)

for j in range(n_intervals):
    lat_min = -90 + j * 5
    lat_max = -90 + (j + 1) * 5
    mask = (lat_flat >= lat_min) & (lat_flat < lat_max)
    
    if np.any(mask):
        eta_lat[j] = np.nanmean(eta_flat[mask])
        eta_std[j] = np.nanstd(eta_flat[mask])

eta_lat = eta_lat * 100
eta_std = eta_std * 100

# ==== 4. Plot ====
fig, ax = plt.subplots(figsize=(1.5, 3))
ax.set_facecolor('white')

# Mean curve
ax.plot(eta_lat, range(1, n_intervals + 1), 
        color=[0.0431, 0.4627, 0.2039], 
        linewidth=1.2)

# Standard deviation shadow
if np.nanmax(eta_std) > 0.1:
    x_lower = np.maximum(eta_lat - eta_std, 0)
    x_upper = eta_lat + eta_std
    
    y_order = np.concatenate([range(1, n_intervals + 1), range(n_intervals, 0, -1)])
    x_lower_order = np.concatenate([x_lower, x_lower[::-1]])
    x_upper_order = np.concatenate([x_upper, x_upper[::-1]])
    
    ax.fill_betweenx(y_order, x_lower_order, x_upper_order,
                     alpha=0.2, 
                     color=[0.5, 0.5, 0.5], 
                     edgecolor='none')

# Reference lines
ax.axvline(x=100, linestyle='--', linewidth=0.8, color=[7/255, 7/255, 7/255])
ax.axvline(x=0, linestyle='--', linewidth=0.8, color=[7/255, 7/255, 7/255])

# Axis limits
ax.set_ylim(-0.5, n_intervals + 1.5)
ax.set_xlim(-5, 105)

# Y-axis latitude labels
key_lats = [-90, -60, -30, 0, 30, 60, 90]
key_y_pos = [(lat + 90) / 5 + 0.5 for lat in key_lats]
ax.set_yticks(key_y_pos)
ax.set_yticklabels([f'{abs(lat)}°S' if lat < 0 else f'{lat}°N' if lat > 0 else '0°' for lat in key_lats])

# X-axis
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_xlabel('Ratio (%)', fontsize=10)

plt.tight_layout()

# Save figure
fig.savefig(
    r"C:\Users\Lenovo\Desktop\LatitudeRatio.png",
    dpi=500,
    bbox_inches='tight',
    facecolor='white',
    edgecolor='none'
)
plt.show()